# Avon River analysis — 03 Hypothesis tests and sensitivity analysis

Eric Gomez · October 2026

The hypotheses below were stated before any test was run. They are tested on the cleaned tables from `02`
(`data/processed/`). Six hypotheses are tested at once, so the p-values are adjusted with Holm's method
(Bruce et al., 2020, ch. 3, "Multiple Testing"). The notebook then checks whether any conclusion depends on
a cleaning judgement call (the sensitivity analysis), and whether a predictive model would add anything.

In [1]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()


from scipy import stats
from src.paths import PROCESSED

visits = pd.read_csv(PROCESSED / 'visits.csv', parse_dates=['date'])
records = pd.read_csv(PROCESSED / 'fish_records.csv', parse_dates=['date'])
records['month'] = records['date'].dt.to_period('M')
print('visits:', len(visits), '| fish records:', len(records))

visits: 69 | fish records: 70


## Hypotheses, stated before testing

| # | Claim | H0 (nothing there) | Data |
|---|---|---|---|
| H1 | Water temperature differs between sites | Same distribution at AV-1/2/3 | 69 visits |
| H2 | Dissolved oxygen differs between sites | Same distribution | 69 visits |
| H3 | Īnanga counts per survey differ between months | Same distribution in Oct, Nov and Dec | 25 records |
| H4 | Īnanga counts are associated with oxygen saturation | ρ = 0 | 25 records |
| H5 | Longfin eel counts are associated with oxygen saturation | ρ = 0 | 15 |
| H6 | Shortfin eel counts are associated with oxygen saturation | ρ = 0 | 11 |

- **Direction.** All tests are two-sided: no direction is assumed before seeing the data.
- **α and the correction.** α = 0.05, **Holm-adjusted across all six**. A false "relationship" could send
  conservation money the wrong way, so the family-wise error rate is controlled.

## Test choice and assumptions

- **H1–H3: Kruskal–Wallis**, three or more groups and rank-based.
  - Chosen because counts are skewed, groups are small and unequal (6–25), and normality can't be
    assumed.
  - Effect size: ε² = (H − k + 1)/(n − k).
- **H4–H6: Spearman's ρ.** Monotonic and rank-based, for the same reasons (see *Correlation method* in `02`).
- **Assumption not fully met: independence.** Visits to one site are repeated over time, so they are not
  fully independent. The p-values are therefore approximate.
- **H3 gets a permutation re-check.** It is the result closest to the threshold, so its p-value is
  re-checked by shuffling the month labels 10,000 times. That makes no distributional assumption.

In [2]:
def kruskal(groups):
    groups = [np.asarray(g, float) for g in groups if len(g)]
    h, p = stats.kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    return h, p, (h - k + 1) / (n - k), n


def holm(p):
    """Holm's step-down adjustment: walk up the sorted p-values, multiplying by the number still in play."""
    p = np.asarray(p, float); adj = np.empty(len(p)); running = 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adj[i] = running
    return adj


def run_tests(visits, records):
    """H1-H6 on one version of the cleaned data, Holm-adjusted across the six."""
    tests = []
    h, p, e2, n = kruskal([g.temperature_c for _, g in visits.groupby('site_id')])
    tests.append(('H1', 'temperature by site', 'Kruskal–Wallis H', h, 'ε²', e2, n, p))
    h, p, e2, n = kruskal([g.do_mg_l.dropna() for _, g in visits.groupby('site_id')])
    tests.append(('H2', 'DO by site', 'Kruskal–Wallis H', h, 'ε²', e2, n, p))
    ina = records[records.species == 'Inanga']
    h, p, e2, n = kruskal([g['count'] for _, g in ina.groupby('month')])
    tests.append(('H3', 'īnanga count by month', 'Kruskal–Wallis H', h, 'ε²', e2, n, p))
    for hid, sp in [('H4', 'Inanga'), ('H5', 'Longfin Eel'), ('H6', 'Shortfin Eel')]:
        d = records[records.species == sp].dropna(subset=['pct_sat'])
        rho, p = stats.spearmanr(d.pct_sat, d['count'])
        tests.append((hid, f'{sp.lower()} count ~ % saturation', 'Spearman ρ', rho, 'ρ', rho, len(d), p))
    res = pd.DataFrame(tests, columns=['id', 'hypothesis', 'statistic', 'value', 'effect size', 'effect', 'n', 'p'])
    res['p (Holm)'] = holm(res.p)
    res['reject H0 at 0.05'] = res['p (Holm)'] < 0.05
    return res


res = run_tests(visits, records)
from src.visualization.presentation import save_table
out = res.assign(value=res.value.round(3), effect=res.effect.round(3), p=res.p.map('{:.2g}'.format),
                 **{'p (Holm)': res['p (Holm)'].map('{:.2g}'.format)})
display(out)
_ = save_table(out, '03_hypothesis_tests')
ina = records[records.species == 'Inanga']                # used by the permutation re-check below

,id,hypothesis,statistic,value,effect size,effect,n,p,p (Holm),reject H0 at 0.05
0,H1,temperature by site,Kruskal–Wallis H,21.512,ε²,0.296,69,2.1e-05,0.00013,True
1,H2,DO by site,Kruskal–Wallis H,3.692,ε²,0.026,69,0.16,0.63,False
2,H3,īnanga count by month,Kruskal–Wallis H,10.365,ε²,0.380,25,0.0056,0.028,True
3,H4,inanga count ~ % saturation,Spearman ρ,-0.163,ρ,-0.163,25,0.44,1,False
4,H5,longfin eel count ~ % saturation,Spearman ρ,0.196,ρ,0.196,15,0.48,1,False
5,H6,shortfin eel count ~ % saturation,Spearman ρ,0.273,ρ,0.273,11,0.42,1,False


saved outputs/tables/03_hypothesis_tests.csv

id                        hypothesis        statistic  value effect size  effect  n       p p (Holm)  reject H0 at 0.05
H1               temperature by site Kruskal–Wallis H 21.512          ε²   0.296 69 2.1e-05  0.00013               True
H2                        DO by site Kruskal–Wallis H  3.692          ε²   0.026 69    0.16     0.63              False
H3             īnanga count by month Kruskal–Wallis H 10.365          ε²   0.380 25  0.0056    0.028               True
H4       inanga count ~ % saturation       Spearman ρ -0.163           ρ  -0.163 25    0.44        1              False
H5  longfin eel count ~ % saturation       Spearman ρ  0.196           ρ   0.196 15    0.48        1              False
H6 shortfin eel count ~ % saturation       Spearman ρ  0.273           ρ   0.273 11    0.42        1              False


In [3]:
# H3 re-check without distributional assumptions: shuffle the month labels among the īnanga records
# and count how often chance alone gives an H at least as large as the one observed.
months_arr, counts_arr = ina['month'].to_numpy(), ina['count'].to_numpy()
labels = np.unique(months_arr)
observed_h = stats.kruskal(*[counts_arr[months_arr == m] for m in labels]).statistic
rng = np.random.default_rng(803)
null_h = np.array([stats.kruskal(*[counts_arr[shuffled == m] for m in labels]).statistic
                   for shuffled in (rng.permutation(months_arr) for _ in range(10_000))])
p_perm = (np.sum(null_h >= observed_h) + 1) / (len(null_h) + 1)
print(f'H3: observed H = {observed_h:.2f}; permutation p = {p_perm:.4f} (10,000 shuffles); '
      f'with the Holm factor for its rank (x5) = {min(1, 5 * p_perm):.3f}')

H3: observed H = 10.37; permutation p = 0.0023 (10,000 shuffles); with the Holm factor for its rank (x5) = 0.011


### Interpretation

- **H1 holds.** Temperature differs between sites (Holm p < 0.001, large effect). AV-3 is warmest.
- **H3 holds.** Īnanga counts differ between months (Holm p ≈ 0.03, ε² ≈ 0.38).
  - The permutation re-check agrees: p ≈ 0.002, or about 0.01 after the Holm factor.
  - So the December drop is more than chance would usually produce, even after correcting for six tests.
- **H2, H4, H5 and H6 are not supported.** DO does not differ detectably between sites, and no native
  species' count is associated with oxygen saturation.
- **What the data cannot show.** "Low oxygen drives fish numbers" is **not shown** by these data. At 11–25
  records per species, only a large effect could have been detected. Settling the question needs more
  surveys, with sampling times and survey effort recorded (see *Open questions* in `02`).

## Sensitivity analysis: do the conclusions survive each judgement call?

`02` rebuilt the cleaned tables under every alternative to the four judgement calls in `01` (scenarios saved
in `data/interim/sensitivity/`). The cell below recomputes the headline numbers and re-runs H1–H6 on each.
A conclusion that changed under a defensible alternative would have to be reported as uncertain.

In [4]:
from src.paths import INTERIM

def headline(visits, records):
    """The headline numbers for one version of the cleaned data."""
    by_month = (visits.assign(month=visits.date.dt.to_period('M'))
                      .groupby(['month', 'site_id']).temperature_c.median().unstack())
    ina_month = records[records.species == 'Inanga'].groupby('month')['count'].mean()
    december = visits[visits.date.dt.month == 12]
    return {'visits': len(visits), 'fish records': len(records),
            'December mean temperature (°C)': f'{december.temperature_c.mean():.2f}',
            'AV-3 warmest every month': bool((by_month.idxmax(axis=1) == 'AV-3').all()),
            'visits below saturation': f'{int((visits.pct_sat < 100).sum())} of {int(visits.pct_sat.notna().sum())}',
            'visits below 7 mg/L': int((visits.do_mg_l < 7).sum()),
            'every site below 7 mg/L at least once': bool((visits.groupby('site_id').do_mg_l.min() < 7).all()),
            'īnanga per survey, Oct / Nov / Dec': ' / '.join(f'{x:.1f}' for x in ina_month)}

scenario_dir = INTERIM / 'sensitivity'
index = pd.read_csv(scenario_dir / 'index.csv')
table, verdicts = {}, {}
for i, name in enumerate(index.scenario):
    v = pd.read_csv(scenario_dir / f'{i}_visits.csv', parse_dates=['date'])
    r = pd.read_csv(scenario_dir / f'{i}_records.csv', parse_dates=['date'])
    r['month'] = r['date'].dt.to_period('M')
    t = run_tests(v, r).set_index('id')
    h = headline(v, r)
    table[name] = {**h, **{f'{k} p (Holm)': f"{t.loc[k, 'p (Holm)']:.2g}" + (' *' if t.loc[k, 'reject H0 at 0.05'] else '')
                         for k in t.index}}
    verdicts[name] = (tuple(t['reject H0 at 0.05']), h['AV-3 warmest every month'],
                      h['every site below 7 mg/L at least once'])
sens = pd.DataFrame(table)
sens.loc['same conclusions as chosen'] = [verdicts[s] == verdicts['chosen'] for s in sens.columns]
display(sens)
print('* = H0 rejected at 0.05 after Holm adjustment')
_ = save_table(sens.reset_index().rename(columns={'index': 'result'}), '03_sensitivity')

,chosen,same-day: first,row 64 = 12 Dec,keep row 94,drop row 45,64 + 94 + 45
visits,69,69,70,69,69,70
fish records,70,70,70,70,69,69
December mean temperature (°C),18.11,18.08,18.23,18.14,18.11,18.26
AV-3 warmest every month,True,True,True,True,True,True
visits below saturation,69 of 69,69 of 69,70 of 70,69 of 69,69 of 69,70 of 70
visits below 7 mg/L,12,12,12,11,12,11
every site below 7 mg/L at least once,True,True,True,True,True,True
"īnanga per survey, Oct / Nov / Dec",22.5 / 23.0 / 10.8,22.5 / 23.0 / 10.8,22.5 / 23.0 / 10.8,22.5 / 23.0 / 10.8,22.0 / 23.0 / 10.8,22.0 / 23.0 / 10.8
H1 p (Holm),0.00013 *,0.00013 *,7.5e-05 *,9.7e-05 *,0.00013 *,5.6e-05 *
H2 p (Holm),0.63,0.72,0.59,1,0.63,1


* = H0 rejected at 0.05 after Holm adjustment
saved outputs/tables/03_sensitivity.csv

                               result             chosen    same-day: first    row 64 = 12 Dec        keep row 94        drop row 45       64 + 94 + 45
                               visits                 69                 69                 70                 69                 69                 70
                         fish records                 70                 70                 70                 70                 69                 69
       December mean temperature (°C)              18.11              18.08              18.23              18.14              18.11              18.26
             AV-3 warmest every month               True               True               True               True               True               True
              visits below saturation           69 of 69           69 of 69           70 of 70           69 of 69           69 of 69           70 of 70
 

### Interpretation

- **Every conclusion survives every alternative.** In all six versions the same hypotheses are rejected
  (H1 and H3) and the same are not (H2, H4–H6). AV-3 is warmest every month, and every site dips below
  7 mg/L at least once.
- **What moves, and by how much:**
  - December's mean temperature: 18.08–18.26 °C.
  - Visits below 7 mg/L: 11 or 12. Keeping row 94 lifts the AV-3 visit on 29 Dec above 7 mg/L.
  - H3's Holm p: 0.028–0.032. The īnanga result is the one closest to the threshold, and it holds.
- **Conclusion.** None of the cleaning choices that needed judgement drives a finding.

## Is a predictive model justified?

- **The tests.** H4–H6 found no association between oxygen saturation and native fish counts.
- **The out-of-sample check.** A model only earns its place if it predicts better than the mean. Below,
  `compare_models` (`src/models/baseline.py`) fits *count ~ % saturation* for each native species and
  compares its leave-one-out error with a model that always predicts the mean.
- **If none beats the baseline,** fitting a more flexible model (Poisson regression, a decision tree) to
  these data would only fit noise.

In [5]:
from src.models.baseline import compare_models

summary = []
for sp in ['Inanga', 'Longfin Eel', 'Shortfin Eel']:
    d = records[records.species == sp]
    print('\n' + '=' * 30, sp, '=' * 30)
    res = compare_models(d, 'pct_sat', 'count', degrees=(1,))
    if res:
        lin = res['results'][0]
        summary.append({'species': sp, 'n': res['n'], 'LOOCV RMSE (linear)': lin['loocv'],
                        'LOOCV RMSE (mean baseline)': res['baseline_loocv'],
                        'beats baseline': lin['loocv'] < res['baseline_loocv']})
summary = pd.DataFrame(summary)
display(summary)
_ = save_table(summary, '03_model_vs_baseline')


============================== Inanga ==============================
count ~ pct_sat   (n = 25 complete cases)
--------------------------------------------------------------------
model                   R2 (train)    adj R2    LOOCV RMSE
--------------------------------------------------------------------
linear                      0.0095   -0.0335        7.4307
mean-only baseline                                  7.1256
--------------------------------------------------------------------
  linear: count = 28.3950 - 0.1037 * pct_sat

Reading these numbers:
  R2 (train) rises with flexibility whether or not the extra flexibility
  is real, so it cannot rank models. LOOCV RMSE is out-of-sample: lower
  is better, and anything above the baseline is worse than the mean.
  No model here beats the mean baseline (7.1256).

This function does not choose. Record your choice and your reason.

============================== Longfin Eel ==============================
count ~ pct_sat   (n = 15 co

,species,n,LOOCV RMSE (linear),LOOCV RMSE (mean baseline),beats baseline
0,Inanga,25,7.4307,7.1256,False
1,Longfin Eel,15,4.8918,4.8550,False
2,Shortfin Eel,11,1.7817,1.6553,False


saved outputs/tables/03_model_vs_baseline.csv

     species  n  LOOCV RMSE (linear)  LOOCV RMSE (mean baseline)  beats baseline
      Inanga 25               7.4307                      7.1256           False
 Longfin Eel 15               4.8918                      4.8550           False
Shortfin Eel 11               1.7817                      1.6553           False


### Interpretation

- **No model beats the mean.** For no native species does a model using oxygen saturation predict counts
  better than simply predicting the average; its leave-one-out error is the same or worse.
- **So no predictive model is fitted.** Count regression and classification suit this question, but they
  need multi-year data with survey effort recorded, which the organisation is said to hold. That is a
  finding in its own right, not a gap to hide.

## Summary of findings

- **Robust to the cleaning judgement calls:** no alternative in the sensitivity analysis changes a conclusion.
- **Supported:**
  - sites differ in temperature, and the river warmed from October to December;
  - īnanga counts per survey fell in December.
- **Not supported at these sample sizes:**
  - DO differs between sites;
  - fish counts track oxygen saturation.
- **Not testable with these data:**
  - causes;
  - daily minima;
  - summer conditions;
  - effort-adjusted abundance.

## References

Bruce, P., Bruce, A., & Gedeck, P. (2020). *Practical statistics for data scientists: 50+ essential concepts
using R and Python* (2nd ed.). O'Reilly Media.